In [0]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import torch
import wandb
import time

import sys
import os
sys.path.append(os.path.abspath("../../"))
from QNeuralForecast.dlinear import QDLinear
from QNeuralForecast.patchtst import QPatchTST
from QNeuralForecast.timesnet import QTimesNet
from QNeuralForecast.nhits import QNHITS
from neuralforecast.models import DLinear, NHITS, PatchTST, TimesNet

from neuralforecast import NeuralForecast
from pytorch_lightning.loggers import WandbLogger
from neuralforecast.losses.numpy import mae, mse, rmse, mape, smape, mase

def set_global_seed(seed: int):
    """Fix all global random sources for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [0]:
os.environ["WANDB_API_KEY"] = dbutils.secrets.get(scope="wandb-scope", key="wandb-api-key")
wandb.login()

In [0]:
df = pd.read_csv("../data/train_prepared.csv", parse_dates=["Date"], low_memory=False)
df = df.sort_values(["Store", "Date"])

delta_days = (max(df["Date"]) - min(df["Date"])).days
train_days = int(delta_days * 0.8)
split_date = min(df["Date"]) + np.timedelta64(train_days,"D")

train_df = df[df["Date"] < split_date].copy()
test_df  = df[df["Date"] >= split_date].copy()

HORIZON = (max(df["Date"]) - np.datetime64(split_date)).days

In [0]:
# ── Experiment label ─────────────────────────────────────────────────────────
# Change this before each run to keep results separated in W&B.
# e.g. "dev", "hpo_test", "final"
EXPERIMENT = "dev"

# --- Classical models (no circuit_device param) — trained once as baseline ---
CLASSICAL_MODEL_CLASSES = [DLinear]
# CLASSICAL_MODEL_CLASSES = [DLinear, NHITS, PatchTST, TimesNet]

# --- Quantum models (have circuit_device param) — trained once per device ---
QUANTUM_MODEL_CLASSES = []
#QUANTUM_MODEL_CLASSES = [QDLinear, QNHITS, QPatchTST, QTimesNet]

# --- Devices to compare ---
# Each key becomes a run label in results; value is passed as circuit_device=...
QUANTUM_DEVICES = {
    "ideal":     "default.qubit",   # noise-free statevector simulation
    "noisy_sim": "default.mixed",   # mixed-state sim — add depolarising noise in circuit
    # "real_nisq": "qiskit.ibm.remote",  # actual hardware (requires IBM account + credentials)
}

# Combined for HPO (shared best_params applied to all model types)
model_classes = CLASSICAL_MODEL_CLASSES + QUANTUM_MODEL_CLASSES

In [0]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
# --- Hyperparameter search space ---
HYPERPARAM_GRID = {
    "input_size":    [100, 200, 400],
    "max_steps":     [50, 100, 300],
    "learning_rate": (1e-4, 1e-1),   # log-uniform range
    "scaler_type":   ["standard", "robust", "minmax", "identity"],
}

def objective(trial):
    set_global_seed(trial.number)
    input_size    = trial.suggest_categorical("input_size",    HYPERPARAM_GRID["input_size"])
    max_steps     = trial.suggest_categorical("max_steps",     HYPERPARAM_GRID["max_steps"])
    learning_rate = trial.suggest_float("learning_rate", *HYPERPARAM_GRID["learning_rate"], log=True)
    scaler_type   = trial.suggest_categorical("scaler_type",   HYPERPARAM_GRID["scaler_type"])

    trial_models = [
        cls(
            h=HORIZON,
            input_size=input_size,
            max_steps=max_steps,
            learning_rate=learning_rate,
            scaler_type=scaler_type,
            start_padding_enabled=True,
            # Disable logging during HPO — Optuna tracks CV MAE, training curves not needed
            logger=False,
        )
        for cls in model_classes
    ]

    nf_trial = NeuralForecast(models=trial_models, freq="D")
    cv = nf_trial.cross_validation(
        train_df, n_windows=4, step_size=100,
        verbose=False, refit=False,
        id_col="Store", time_col="Date", target_col="Sales"
    )
    model_cols = [c for c in cv.columns if c not in ["Store", "Date", "Sales", "cutoff"]]
    avg_mae = float(np.mean([mae(cv["Sales"].values, cv[m].values) for m in model_cols]))
    return avg_mae

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="rossmann_hpo",
)
study.optimize(objective, n_trials=20, n_jobs=1, show_progress_bar=True)

best_params = study.best_params
print(f"\nBest CV MAE : {study.best_value:.4f}")
print(f"Best params : {best_params}")

In [0]:
# --- HPO results summary ---
print(f"Best CV MAE : {study.best_value:.4f}")
print(f"Best params : {study.best_params}\n")

trials_df = study.trials_dataframe().sort_values("value")
print(trials_df[["number", "value", "params_input_size", "params_max_steps",
                  "params_learning_rate", "params_scaler_type"]].to_string(index=False))

# Plot trial history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot([t.number for t in study.trials],
             [t.value for t in study.trials], marker='o', linewidth=1)
axes[0].axhline(study.best_value, color='red', linestyle='--', label=f"Best: {study.best_value:.2f}")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("CV MAE")
axes[0].set_title("Optuna — Trial history")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()))
axes[1].set_xlabel("Importance")
axes[1].set_title("Hyperparameter importances")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
SEEDS = [0, 1, 2]
#SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# all_seed_forecasts  : {run_label -> [seed_fc_df, ...]}
# all_training_times  : {run_label -> [elapsed_sec_per_seed, ...]}
# all_param_counts    : {run_label -> {model_name -> n_params}}  (same for every seed)
all_seed_forecasts = {}
all_training_times = {}
all_param_counts   = {}


def count_parameters(nf_fitted):
    """Return {model_class_name: trainable_param_count} for every model in a fitted NF."""
    counts = {}
    for m in nf_fitted.models:
        name = m.__class__.__name__
        n = sum(p.numel() for p in m.parameters() if p.requires_grad)
        counts[name] = n
    return counts


# ── 0. Naive baseline (lag-1: forecast[t] = actual[t-1]) ────────────────────
print("═" * 60)
print("NAIVE BASELINE  (lag-1: forecast[t] = actual[t-1])")
print("═" * 60)
all_dates = (
    pd.concat([train_df[["Store", "Date", "Sales"]],
               test_df [["Store", "Date", "Sales"]]])
    .sort_values(["Store", "Date"])
    .reset_index(drop=True)
)
naive_all = (
    all_dates
    .groupby("Store", group_keys=False)
    .apply(lambda g: g.assign(Naive=g["Sales"].shift(1)))
)
naive_fc = (
    naive_all
    .merge(test_df[["Store", "Date"]], on=["Store", "Date"], how="inner")
    [["Store", "Date", "Naive"]]
    .reset_index(drop=True)
)
all_seed_forecasts["naive"] = [naive_fc]
all_training_times["naive"] = [0.0]          # no trainable model
all_param_counts  ["naive"] = {"Naive": 0}
print(f"  Forecast range: {naive_fc['Date'].min().date()} → {naive_fc['Date'].max().date()}")
print(f"  ✓ Naive: done.")

# ── 1. Classical models (device-independent, trained once) ──────────────────
if CLASSICAL_MODEL_CLASSES:
    print("═" * 60)
    print("CLASSICAL MODELS")
    print("═" * 60)
    classical_forecasts = []
    classical_times     = []
    for seed in SEEDS:
        print(f"\n  --- seed {seed} ---")
        set_global_seed(seed)
        seed_models = [
            cls(
                h=HORIZON,
                input_size=best_params["input_size"],
                max_steps=best_params["max_steps"],
                learning_rate=best_params["learning_rate"],
                scaler_type=best_params["scaler_type"],
                start_padding_enabled=True,
                random_seed=seed,
                early_stop_patience_steps=5,
                logger=WandbLogger(
                    project="qtsf",
                    name=f"{EXPERIMENT}/classical/{cls.__name__}/seed_{seed}",
                    group=f"{EXPERIMENT}/classical/{cls.__name__}",
                    tags=[EXPERIMENT, "classical", "training", cls.__name__],
                ),
            )
            for cls in CLASSICAL_MODEL_CLASSES
        ]
        nf = NeuralForecast(models=seed_models, freq="D")
        t0 = time.perf_counter()
        nf.fit(train_df, id_col="Store", time_col="Date", target_col="Sales", val_size=HORIZON)
        elapsed = time.perf_counter() - t0
        classical_times.append(elapsed)
        if seed == SEEDS[0]:
            all_param_counts["classical"] = count_parameters(nf)
        fc = nf.predict().reset_index()
        wandb.finish()
        classical_forecasts.append(fc)
        print(f"  Training time  : {elapsed:.1f}s")
        print(f"  Forecast range : {fc['Date'].min().date()} → {fc['Date'].max().date()}")
    all_seed_forecasts["classical"] = classical_forecasts
    all_training_times["classical"] = classical_times
    print(f"\n  ✓ Classical: {len(SEEDS)} seeds done  "
          f"(mean time {np.mean(classical_times):.1f}s ± {np.std(classical_times):.1f}s)")
    print(f"  Param counts: {all_param_counts['classical']}")

# ── 2. Quantum models (one seed loop per device) ────────────────────────────
if QUANTUM_MODEL_CLASSES:
    for dev_label, dev_name in QUANTUM_DEVICES.items():
        print("\n" + "═" * 60)
        print(f"QUANTUM MODELS  |  device: {dev_label}  ({dev_name})")
        print("═" * 60)
        dev_forecasts = []
        dev_times     = []
        for seed in SEEDS:
            print(f"\n  --- seed {seed} ---")
            set_global_seed(seed)
            seed_models = [
                cls(
                    h=HORIZON,
                    input_size=best_params["input_size"],
                    max_steps=best_params["max_steps"],
                    learning_rate=best_params["learning_rate"],
                    scaler_type=best_params["scaler_type"],
                    start_padding_enabled=True,
                    random_seed=seed,
                    early_stop_patience_steps=5,
                    circuit_device=dev_name,
                    logger=WandbLogger(
                        project="qtsf",
                        name=f"{EXPERIMENT}/{dev_label}/{cls.__name__}/seed_{seed}",
                        group=f"{EXPERIMENT}/{dev_label}/{cls.__name__}",
                        tags=[EXPERIMENT, dev_label, "training", cls.__name__],
                    ),
                )
                for cls in QUANTUM_MODEL_CLASSES
            ]
            nf = NeuralForecast(models=seed_models, freq="D")
            t0 = time.perf_counter()
            nf.fit(train_df, id_col="Store", time_col="Date", target_col="Sales", val_size=HORIZON)
            elapsed = time.perf_counter() - t0
            dev_times.append(elapsed)
            if seed == SEEDS[0]:
                all_param_counts[dev_label] = count_parameters(nf)
            fc = nf.predict().reset_index()
            wandb.finish()
            dev_forecasts.append(fc)
            print(f"  Training time  : {elapsed:.1f}s")
            print(f"  Forecast range : {fc['Date'].min().date()} → {fc['Date'].max().date()}")
        all_seed_forecasts[dev_label] = dev_forecasts
        all_training_times[dev_label] = dev_times
        print(f"\n  ✓ {dev_label}: {len(SEEDS)} seeds done  "
              f"(mean time {np.mean(dev_times):.1f}s ± {np.std(dev_times):.1f}s)")
        print(f"  Param counts: {all_param_counts[dev_label]}")

print(f"\nAll runs complete. Run labels: {list(all_seed_forecasts.keys())}")

In [0]:
y_train_global = train_df["Sales"].values
SEASONALITY = 7

# all_avg_results  : {run_label -> {model -> {metric -> float}}}  (seed-ensemble score)
# all_seed_results : {run_label -> {model -> {metric -> [per-seed floats]}}}
all_avg_results  = {}
all_seed_results = {}
all_avg_forecasts = {}  # {run_label -> avg_forecast_df}  — kept for store plots

for run_label, seed_forecasts in all_seed_forecasts.items():

    # --- Average forecasts across seeds ---
    model_names = [c for c in seed_forecasts[0].columns if c not in ["index", "Store", "Date"]]
    avg_forecasts = seed_forecasts[0][["Store", "Date"]].copy()
    for model in model_names:
        avg_forecasts[model] = np.mean([fc[model].values for fc in seed_forecasts], axis=0)
    all_avg_forecasts[run_label] = avg_forecasts

    comparison = test_df[["Store", "Date", "Sales"]].merge(
        avg_forecasts, on=["Store", "Date"], how="inner"
    )

    # --- Per-seed metrics ---
    seed_results = {m: {k: [] for k in ["MAE", "MSE", "RMSE", "MAPE", "MASE"]}
                    for m in model_names}
    for fc in seed_forecasts:
        comp_s = test_df[["Store", "Date", "Sales"]].merge(fc, on=["Store", "Date"], how="inner")
        for model in model_names:
            seed_results[model]["MAE" ].append(mae( comp_s["Sales"], comp_s[model]))
            seed_results[model]["MSE" ].append(mse( comp_s["Sales"], comp_s[model]))
            seed_results[model]["RMSE"].append(rmse(comp_s["Sales"], comp_s[model]))
            seed_results[model]["MAPE"].append(mape(comp_s["Sales"], comp_s[model]))
            seed_results[model]["MASE"].append(mase(comp_s["Sales"], comp_s[model],
                                                    y_train=y_train_global,
                                                    seasonality=SEASONALITY))

    # --- Ensemble (seed-averaged forecast) metrics ---
    avg_results = {}
    for model in model_names:
        avg_results[model] = {
            "MAE" : mae( comparison["Sales"], comparison[model]),
            "MSE" : mse( comparison["Sales"], comparison[model]),
            "RMSE": rmse(comparison["Sales"], comparison[model]),
            "MAPE": mape(comparison["Sales"], comparison[model]),
            "MASE": mase(comparison["Sales"], comparison[model],
                         y_train=y_train_global, seasonality=SEASONALITY),
        }

    all_avg_results [run_label] = avg_results
    all_seed_results[run_label] = seed_results

    # --- Training-time stats for this run label ---
    run_times = all_training_times.get(run_label, [])
    mean_training_time = float(np.mean(run_times)) if run_times else 0.0
    std_training_time  = float(np.std(run_times))  if run_times else 0.0

    # --- W&B logging (one run per model x device combination) ---
    for model in model_names:
        run = wandb.init(
            project="qtsf",
            name=f"{EXPERIMENT}/{run_label}/{model}",
            group=f"{EXPERIMENT}/{run_label}",
            tags=[EXPERIMENT, run_label, model],
        )
        log_dict: dict = {"model": model, "run_label": run_label, "experiment": EXPERIMENT}

        # Accuracy metrics
        for metric_name, val in avg_results[model].items():
            log_dict[f"avg_{metric_name}"] = float(val)
        for metric_name, vals in seed_results[model].items():
            log_dict[f"mean_{metric_name}"] = float(np.mean(vals))
            log_dict[f"std_{metric_name}"]  = float(np.std(vals))

        # Model size
        param_count = all_param_counts.get(run_label, {}).get(model, None)
        if param_count is not None:
            log_dict["num_parameters"] = param_count

        # Training time (shared cost across all models in this run's seed loop)
        log_dict["mean_training_time_s"] = mean_training_time
        log_dict["std_training_time_s"]  = std_training_time

        run.log(log_dict)
        run.finish()


In [0]:
# ── Summary table ────────────────────────────────────────────────────────────
print("\nTest-set metrics  (seed-ensemble ★ | seeds mean ± std)")
print("─" * 70)
for run_label in all_avg_results:
    print(f"\n  [{run_label}]")
    for model in sorted(all_avg_results[run_label], key=lambda m: all_avg_results[run_label][m]["MAE"]):
        a = all_avg_results[run_label][model]
        s = all_seed_results[run_label][model]
        print(f"    {model}:")
        for metric in ["MAE", "MSE", "RMSE", "MAPE", "MASE"]:
            print(f"      {metric}: ★{a[metric]:.4f}  |  "
                  f"seeds {np.mean(s[metric]):.4f} ± {np.std(s[metric]):.4f}")

In [0]:
# ── Box plots: one row per run_label, one subplot per metric ─────────────────
metrics_to_plot = ["MAE", "MSE", "RMSE", "MAPE", "MASE"]
n_runs = len(all_seed_results)

fig, axes = plt.subplots(n_runs, len(metrics_to_plot),
                         figsize=(4 * len(metrics_to_plot), 4 * n_runs),
                         squeeze=False)
fig.suptitle(
    "Per-seed metrics (box/dots) vs. seed-averaged forecast (★)\n"
    "Rows = run (classical / quantum device),  Columns = metric",
    fontsize=11, y=1.01
)

for row, (run_label, seed_results) in enumerate(all_seed_results.items()):
    model_names = list(seed_results.keys())
    for col, metric in enumerate(metrics_to_plot):
        ax = axes[row][col]
        data = [seed_results[m][metric] for m in model_names]
        ax.boxplot(data, tick_labels=model_names, patch_artist=True, notch=False,
                   medianprops=dict(color="black", linewidth=2))
        for i, values in enumerate(data, start=1):
            ax.scatter([i] * len(values), values, color="black", zorder=5, s=20,
                       label="individual seed" if (i == 1 and col == 0) else "")
        for i, m in enumerate(model_names, start=1):
            ax.scatter(i, all_avg_results[run_label][m][metric],
                       marker="*", color="red", zorder=6, s=150,
                       label="seed-avg ★" if (i == 1 and col == 0) else "")
        if col == 0:
            ax.set_ylabel(f"{run_label}\n{metric}", fontsize=9)
        else:
            ax.set_ylabel(metric, fontsize=9)
        if row == 0:
            ax.set_title(metric)
        ax.grid(True, axis="y", alpha=0.3)
        if col == 0:
            ax.legend(loc="upper right", fontsize=7)

plt.tight_layout()
plt.show()

In [0]:
# ── MAE comparison bar chart across all runs ─────────────────────────────────
fig, ax = plt.subplots(figsize=(max(8, 2 * len(all_avg_results)), 4))
x = np.arange(len(all_avg_results))
run_labels = list(all_avg_results.keys())
all_model_names = sorted({m for r in all_avg_results.values() for m in r})
bar_width = 0.8 / len(all_model_names)

for j, model in enumerate(all_model_names):
    maes = [all_avg_results[rl].get(model, {}).get("MAE", float("nan"))
            for rl in run_labels]
    ax.bar(x + j * bar_width, maes, width=bar_width, label=model, alpha=0.85)

ax.set_xticks(x + bar_width * (len(all_model_names) - 1) / 2)
ax.set_xticklabels(run_labels)
ax.set_ylabel("MAE (seed-ensemble)")
ax.set_title("MAE comparison: classical baseline vs. quantum devices")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# ── Helper: compute per-store mean MAE across all models in a set of run_labels ──
def store_mae_for_labels(run_labels_subset):
    """Returns a Series indexed by Store with the mean MAE across all models and runs."""
    per_store = []
    for rl in run_labels_subset:
        avg_fc = all_avg_forecasts[rl]
        model_cols = [c for c in avg_fc.columns if c not in ["Store", "Date"]]
        comp = test_df[["Store", "Date", "Sales"]].merge(avg_fc, on=["Store", "Date"], how="inner")
        for m in model_cols:
            store_err = comp.groupby("Store").apply(
                lambda g: (g["Sales"] - g[m]).abs().mean(), include_groups=False
            )
            per_store.append(store_err)
    return pd.concat(per_store, axis=1).mean(axis=1)


# ── Helper: plot one store's actual vs. all models for a given set of run_labels ──
def plot_store(ax, store_id, run_labels_subset, title):
    store_actual = test_df[test_df["Store"] == store_id].sort_values("Date")
    ax.plot(store_actual["Date"], store_actual["Sales"],
            label="Actual", color="black", linewidth=1.5)
    colors = plt.cm.tab10.colors
    linestyles = ["-", "--", "-.", ":"]
    color_idx = 0
    for r_idx, rl in enumerate(run_labels_subset):
        avg_fc = all_avg_forecasts[rl]
        model_cols = [c for c in avg_fc.columns if c not in ["Store", "Date"]]
        store_fc = avg_fc[avg_fc["Store"] == store_id].sort_values("Date")
        if store_fc.empty:
            continue
        ls = linestyles[r_idx % len(linestyles)]
        for model in model_cols:
            store_mae = (store_actual.set_index("Date")["Sales"]
                         .reindex(store_fc["Date"].values) - store_fc[model].values
                         ).abs().mean()
            ax.plot(store_fc["Date"], store_fc[model],
                    label=f"{rl} / {model}  (MAE {store_mae:.0f})",
                    color=colors[color_idx % len(colors)], linestyle=ls, linewidth=1)
            color_idx += 1
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("Sales")


# ── Identify run label groups ─────────────────────────────────────────────────
classical_labels = [rl for rl in all_avg_forecasts if rl in ("naive", "classical")]
quantum_labels   = [rl for rl in all_avg_forecasts if rl not in ("naive", "classical")]

# ── Find best / worst store per group ────────────────────────────────────────
classical_store_mae = store_mae_for_labels(classical_labels)
best_classical  = classical_store_mae.idxmin()
worst_classical = classical_store_mae.idxmax()
print(f"Classical — best store: {best_classical} (MAE {classical_store_mae[best_classical]:.1f}), "
      f"worst store: {worst_classical} (MAE {classical_store_mae[worst_classical]:.1f})")

if quantum_labels:
    quantum_store_mae = store_mae_for_labels(quantum_labels)
    best_quantum  = quantum_store_mae.idxmin()
    worst_quantum = quantum_store_mae.idxmax()
    print(f"Quantum   — best store: {best_quantum} (MAE {quantum_store_mae[best_quantum]:.1f}), "
          f"worst store: {worst_quantum} (MAE {quantum_store_mae[worst_quantum]:.1f})")

# ── Build figure ──────────────────────────────────────────────────────────────
n_cols = 2
n_rows = 2 if quantum_labels else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 6 * n_rows), squeeze=False)
fig.suptitle("Best & worst store forecasts — classical (top) and quantum (bottom)", fontsize=12)

plot_store(axes[0][0], best_classical,  classical_labels,
           f"Classical+Naive — best store {best_classical}")
plot_store(axes[0][1], worst_classical, classical_labels,
           f"Classical+Naive — worst store {worst_classical}")

if quantum_labels:
    plot_store(axes[1][0], best_quantum,  quantum_labels,
               f"Quantum — best store {best_quantum}")
    plot_store(axes[1][1], worst_quantum, quantum_labels,
               f"Quantum — worst store {worst_quantum}")

plt.tight_layout()
plt.show()
